# Run Inference with Kaggle Models

This tutorial shows how to run inference across the main modalities available on Kaggle using `ONNX.jl` and `Flux.jl`:

- Image classification
- Object detection
- Text / NLP models
- Tabular classification and regression
- Reinforcement learning policies
- Batching for throughput
- GPU acceleration

Prerequisite: you have downloaded a model as shown in **Tutorial 01 — Load a Model**.

In [ ]:
using ONNX, Flux

## 1. Image Classification

Vision models take a `(3, H, W, N)` Float32 tensor (channels, height, width, batch). Resize and normalise your image to match the model's expected input shape before calling the model.

In [ ]:
using Images, ImageTransformations

model = ONNX.load("resnet50.onnx")

# Load and preprocess image
img     = load("photo.jpg")
img_rs  = imresize(img, 224, 224)                          # resize to 224×224
img_f   = Float32.(channelview(img_rs))                    # (3, 224, 224)

# Normalise with ImageNet mean and std
mean_in = reshape(Float32[0.485, 0.456, 0.406], 3, 1, 1)
std_in  = reshape(Float32[0.229, 0.224, 0.225], 3, 1, 1)
img_n   = (img_f .- mean_in) ./ std_in                    # (3, 224, 224)
img_t   = reshape(img_n, 3, 224, 224, 1)                   # add batch dim

out     = model(img_t)
logits  = vec(out["output"])
pred    = argmax(logits)
println("Predicted ImageNet class index: ", pred)

## 2. Object Detection

Detection models return bounding boxes and class scores. The output format varies by architecture — YOLO returns `(cx, cy, w, h, confidence, class_scores...)` per anchor.

In [ ]:
using Images, ImageTransformations

det_model = ONNX.load("yolov8n.onnx")

img   = load("street.jpg")
img_t = let
    rs = imresize(img, 640, 640)
    f  = Float32.(channelview(rs)) ./ 255f0   # normalise to [0, 1]
    reshape(f, 3, 640, 640, 1)
end

out  = det_model(img_t)                        # shape: (85, num_anchors, 1)
preds = out["output0"][:, :, 1]               # (85, num_anchors)

# Filter by confidence
conf_threshold = 0.5f0
keep = findall(i -> preds[5, i] > conf_threshold, 1:size(preds, 2))
println("Detections above threshold: ", length(keep))

for i in keep[1:min(5, end)]
    cx, cy, w, h = preds[1, i], preds[2, i], preds[3, i], preds[4, i]
    cls = argmax(preds[6:end, i])
    println("  class=$(cls)  cx=$(round(cx, digits=1))  cy=$(round(cy, digits=1))  w=$(round(w, digits=1))  h=$(round(h, digits=1))")
end

## 3. NLP — Text Inference

NLP models from Kaggle are typically exported as ONNX from PyTorch or TensorFlow. They expect pre-tokenised integer tensors. Use the tokenizer that matches the model (e.g. from `Transformers.jl`) and pass the token IDs to the ONNX session.

In [ ]:
using Transformers, Transformers.HuggingFace, Flux: softmax

# Load tokenizer from Hugging Face (matches the Kaggle ONNX export)
tkr = HuggingFace.load_tokenizer("distilbert-base-uncased-finetuned-sst-2-english")

# Load the ONNX model downloaded from Kaggle
nlp_model = ONNX.load("distilbert-sst2.onnx")

for text in ["Julia is a fantastic language!", "The documentation is hard to follow."]
    enc    = tkr(text)
    out    = nlp_model(enc.token, enc.attention_mask)
    probs  = softmax(vec(out["logits"]))
    labels = ["NEGATIVE", "POSITIVE"]
    pred   = labels[argmax(probs)]
    println(rpad(text, 50), "  →  ", pred, " (", round(100 * maximum(probs), digits=1), "%)")
end

## 4. Tabular Classification

Tabular models take a feature matrix of shape `(n_features, n_samples)` and return class logits or probabilities. Load your data with `CSV.jl` and convert it to a Float32 matrix.

In [ ]:
using CSV, DataFrames, Flux: softmax

tab_model = ONNX.load("tabular-classifier.onnx")

df = CSV.read("data.csv", DataFrame)
X  = Float32.(Matrix(df[:, 1:end-1]))   # feature columns
X_t = X'                                 # (n_features, n_samples) for ONNX

out   = tab_model(X_t)
probs = softmax(out["probabilities"], dims=1)   # (n_classes, n_samples)
preds = argmax.(eachcol(probs))

println("Predictions for first 10 rows: ", preds[1:min(10, end)])

## 5. Tabular Regression

Regression models output a continuous value per sample. The output tensor has shape `(1, n_samples)`.

In [ ]:
reg_model = ONNX.load("tabular-regressor.onnx")

df = CSV.read("regression_data.csv", DataFrame)
X  = Float32.(Matrix(df[:, 1:end-1]))'

out  = reg_model(X)
preds = vec(out["predictions"])

println("Predicted values (first 5): ", round.(preds[1:5], digits=3))

## 6. Reinforcement Learning — Policy Inference

RL policy models take an observation vector and return action logits or a continuous action. Load the observation, run a forward pass, and select the greedy action.

In [ ]:
policy = ONNX.load("ppo-cartpole.onnx")

# CartPole observation: [cart position, cart velocity, pole angle, pole angular velocity]
obs = Float32[0.02, -0.01, 0.03, -0.02]
obs_t = reshape(obs, :, 1)   # (obs_dim, 1)

out          = policy(obs_t)
action_probs = Flux.softmax(vec(out["action_probs"]))
action       = argmax(action_probs) - 1   # 0-indexed: 0 = push left, 1 = push right

println("Action probabilities: ", round.(action_probs, digits=3))
println("Selected action:      ", action)

## 7. Batching for Throughput

Running many inputs individually has high overhead. Stack them into a single tensor and process in one forward pass.

In [ ]:
using Images, ImageTransformations, BenchmarkTools

model = ONNX.load("resnet50.onnx")

function preprocess(path)
    img = load(path)
    rs  = imresize(img, 224, 224)
    f   = Float32.(channelview(rs))
    mean_in = reshape(Float32[0.485, 0.456, 0.406], 3, 1, 1)
    std_in  = reshape(Float32[0.229, 0.224, 0.225], 3, 1, 1)
    (f .- mean_in) ./ std_in
end

paths  = ["img1.jpg", "img2.jpg", "img3.jpg", "img4.jpg"]
tensors = [preprocess(p) for p in paths]            # each: (3, 224, 224)
batch   = cat(tensors...; dims=4)                    # (3, 224, 224, N)

out    = model(batch)
logits = out["output"]                               # (1000, N)
preds  = argmax.(eachcol(logits))

println("Predicted class indices: ", preds)

## 8. Benchmark Two Models

Compare inference latency between two Kaggle ONNX models on the same input.

In [ ]:
using BenchmarkTools

files = ["resnet50.onnx", "mobilenet_v2.onnx"]

img_t = let
    img = load("photo.jpg")
    rs  = imresize(img, 224, 224)
    f   = Float32.(channelview(rs))
    reshape(f, 3, 224, 224, 1)
end

for f in files
    m = ONNX.load(f)
    t = @benchmark $m($img_t)
    println(rpad(f, 30), "  median: ", round(median(t).time / 1e6, digits=2), " ms")
end

## 9. GPU Acceleration

Move input tensors to GPU with `cu()` from `CUDA.jl`. The ONNX runtime will use the GPU automatically when the inputs are on device.

In [ ]:
using CUDA

if CUDA.functional()
    println("GPU: ", CUDA.name(CUDA.device()))

    model_gpu = ONNX.load("resnet50.onnx")

    img_gpu = cu(img_t)   # move input to GPU
    out_gpu = model_gpu(img_gpu)
    pred    = argmax(vec(Array(out_gpu["output"])))
    println("GPU prediction: ", pred)
else
    println("No CUDA GPU detected — running on CPU")
end

## 10. Extract Intermediate Activations

ONNX.jl lets you request intermediate node outputs by name, which is useful for feature extraction and interpretability.

In [ ]:
using Plots

model = ONNX.load("resnet50.onnx")

# Run with an intermediate output (node name varies by model — inspect graph first)
acts = ONNX.run(model, img_t; intermediate="layer4")

# Plot activation map for the first channel of the first sample
act_map = acts[:, :, 1, 1]
heatmap(act_map, title="Layer 4 activation map", color=:viridis)

## Summary

| Task | Pattern |
|---|---|
| Image classification | `preprocess(img)` → `model(img_t)["output"]` |
| Object detection | `model(img_t)["output0"]` → filter by confidence |
| NLP classification | tokenize → `model(token, mask)["logits"]` |
| Tabular classification | `Float32.(X')` → `model(X_t)["probabilities"]` |
| Tabular regression | `Float32.(X')` → `model(X_t)["predictions"]` |
| RL policy | `reshape(obs, :, 1)` → `softmax(model(obs_t)["action_probs"])` |
| Batching | `cat(tensors...; dims=4)` → single forward pass |
| GPU | `cu(input)` before forward pass |
| Intermediate activations | `ONNX.run(model, input; intermediate="node_name")` |